In [ ]:
# ======================================================
# IMPORTS
# ======================================================
import os
import glob
import pandas as pd
import numpy as np

In [ ]:
# ======================================================
# CONFIGURATION
# ======================================================
DATA_DIR = ""

LABEL_MAP = {
    "white": 0,
    "black": 1,
    "grey": -1,
    "validation": -1
}

In [ ]:
def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(".", "_")
    )
    return df


In [ ]:
def safe_float(x):
    if pd.isna(x):
        return None
    try:
        return float(str(x).replace(",", "."))
    except:
        return None


In [ ]:
def extract_features(df):

    feats = {}

    if "speed" in df.columns:
        s = df["speed"].apply(safe_float).dropna()
        feats["speed_mean"] = s.mean()
        feats["speed_std"]  = s.std()
        feats["speed_max"]  = s.max()
        feats["speed_min"]  = s.min()
    else:
        feats.update({
            "speed_mean": np.nan,
            "speed_std": np.nan,
            "speed_max": np.nan,
            "speed_min": np.nan,
        })

    # ---------- Heading ----------
    if "heading" in df.columns:
        h = df["heading"].apply(safe_float).dropna()
        feats["heading_mean"] = h.mean()
        feats["heading_std"]  = h.std()
    else:
        feats["heading_mean"] = np.nan
        feats["heading_std"]  = np.nan

    # ---------- ROT ----------
    if "rot" in df.columns:
        rot = df["rot"].apply(safe_float).dropna()
        feats["rot_std"] = rot.std()
    else:
        feats["rot_std"] = np.nan

    # stop = booleano ou -1 → sem informação
    if "stop" in df.columns:
        st = df["stop"].apply(safe_float).replace(-1, 0)
        feats["stop_ratio"] = st.mean()
    else:
        feats["stop_ratio"] = np.nan

    feats["n_points"] = len(df)

    return feats


In [ ]:
rows = []

for cls in ["white", "black", "validation", "grey"]:

    folder = f"{DATA_DIR}/{cls}"
    files = glob.glob(folder + "/*.xlsx")

    print(f">>> {cls.upper()} — {len(files)} arquivos")

    for f in files:

        try:

            sheets = pd.ExcelFile(f).sheet_names

            if "Unseenlabs" not in sheets:
                continue


            # --------------------------------------------------
            #  AIS
            # --------------------------------------------------
            df = pd.read_excel(f, sheet_name="Unseenlabs")

            df = normalize_columns(df)


            # --------------------------------------------------
            #  features
            # --------------------------------------------------
            feats = extract_features(df)

            feats["class"] = LABEL_MAP[cls]
            feats["dataset"] = cls
            feats["file"] = os.path.basename(f)


            rows.append(feats)


        except Exception as e:

            print("ERRO:", f, e)


baseline_df = pd.DataFrame(rows)

print("Dataset construído:")
print(baseline_df.shape)

baseline_df.head()

In [ ]:
baseline_df_clean = baseline_df.dropna(thresh=baseline_df.shape[0]*0.5, axis=1)
baseline_df_clean


,speed_mean,speed_std,speed_max,speed_min,heading_mean,heading_std,rot_std,n_points,class,dataset,file
0,6.405957,7.593631,150.0,0.0,191.611232,114.605907,7.340847,39620,0,white,unseenlabsData257761000.xlsx
1,8.106901,6.102272,76.3,0.0,104.176301,117.383471,15.012535,39895,0,white,unseenlabsData258802000.xlsx
2,10.106190,5.948656,131.0,0.0,105.147963,119.344906,2.222509,43118,0,white,unseenlabsData431639000.xlsx
3,8.471774,8.585463,145.0,0.0,180.609472,104.859083,3.009075,41915,0,white,unseenlabsData538010885.xlsx
4,3.997310,5.192320,16.2,0.0,118.142367,123.257756,7.623171,43719,0,white,unseenlabsData247282600.xlsx
5,5.439139,10.707465,158.0,0.0,185.955776,108.625553,9.250422,37604,0,white,unseenlabsData247431600.xlsx
6,1.497149,3.245515,13.0,0.0,94.571235,108.959626,3.233763,46580,0,white,unseenlabsData257099970.xlsx
7,2.562373,5.256341,102.3,0.0,174.234746,83.716987,5.094581,33598,0,white,unseenlabsData247219700.xlsx
8,9.477855,5.326537,127.0,0.0,112.292355,126.742620,8.244320,40921,0,white,unseenlabsData219497000.xlsx
9,6.930787,6.109324,102.3,0.0,102.184365,114.090799,5.694122,42225,0,white,unseenlabsData257450000.xlsx


In [ ]:
df_train = baseline_df_clean[
    baseline_df_clean["dataset"].isin(["white","black"])
].copy()

X = df_train.drop(columns=["class", "file", "dataset"])
y = df_train["class"]

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_imp = imputer.fit_transform(X)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

In [ ]:
X_train = X_scaled
y_train = y

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier

modelos = {

    "LogisticRegression": LogisticRegression(max_iter=2000),

    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ),

    "SVM": SVC(
        probability=True,
        kernel="rbf"
    ),

    "GradientBoosting": GradientBoostingClassifier()

}

modelos_treinados = {}

for nome,modelo in modelos.items():

    modelo.fit(X_train,y_train)

    modelos_treinados[nome] = modelo

    print("Treinado:",nome)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

def avaliar_dataset(modelo,df_dataset):

    X = df_dataset.drop(columns=["class","file","dataset"])

    X = imputer.transform(X)
    X = scaler.transform(X)

    preds = modelo.predict(X)
    probs = modelo.predict_proba(X)[:,1]

    return preds,probs

In [ ]:
df_validation = baseline_df_clean[
    baseline_df_clean["dataset"] == "validation"
].copy()

df_grey = baseline_df_clean[
    baseline_df_clean["dataset"] == "grey"
].copy()

print("Validation:",len(df_validation))
print("Grey:",len(df_grey))

In [ ]:
resultados = []

for nome,modelo in modelos_treinados.items():

    preds,probs = avaliar_dataset(modelo,df_validation)

    y_true = np.ones(len(preds))   # todos são spoofing

    acc = accuracy_score(y_true,preds)
    precision = precision_score(y_true,preds)
    recall = recall_score(y_true,preds)
    f1 = f1_score(y_true,preds)

    cm = confusion_matrix(y_true,preds)

    resultados.append({
        "Modelo":nome,
        "Dataset":"Validation",
        "Accuracy":acc,
        "Precision":precision,
        "Recall":recall,
        "F1":f1
    })

    print("\nModelo:",nome)
    print("Validation Confusion Matrix:")
    print(cm)

In [ ]:
for nome,modelo in modelos_treinados.items():

    preds,probs = avaliar_dataset(modelo,df_grey)

    y_true = np.ones(len(preds))   # todos são spoofing

    acc = accuracy_score(y_true,preds)
    precision = precision_score(y_true,preds)
    recall = recall_score(y_true,preds)
    f1 = f1_score(y_true,preds)

    cm = confusion_matrix(y_true,preds)

    resultados.append({
        "Modelo":nome,
        "Dataset":"Grey",
        "Accuracy":acc,
        "Precision":precision,
        "Recall":recall,
        "F1":f1
    })

    print("\nModelo:",nome)
    print("Grey Confusion Matrix:")
    print(cm)

In [ ]:
df_resultados = pd.DataFrame(resultados)

df_resultados

In [ ]:
OUTPUT = f"{DATA_DIR}/resultado_estudo2_machine_learning.csv"

df_resultados.to_csv(OUTPUT,index=False)

print("Arquivo salvo:",OUTPUT)